# Module 9 • Machine Translation

# Lesson 53 • Transformer-Based Machine Translation and Pretrained Multilingual Models

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Advanced  
**Execution target:** CPU-first; pretrained-model demonstrations are optional

---

## Scope

This lesson moves from recurrent NMT to Transformer-based machine translation and
modern pretrained multilingual systems.

It covers:

- Transformer encoder-decoder MT;
- positional information;
- encoder self-attention;
- masked decoder self-attention;
- encoder-decoder cross-attention;
- teacher forcing and shifted targets;
- greedy decoding;
- beam search concepts;
- pretrained translation models;
- MarianMT;
- M2M-100;
- NLLB;
- language codes and forced target-language tokens;
- zero-shot versus fine-tuned translation;
- multilingual transfer;
- Arabic-English considerations;
- offline evaluation.

The main experiment trains a tiny Transformer from scratch on CPU. The Hugging
Face examples are optional and disabled by default because they require downloads.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain Transformer MT architecture;
- distinguish encoder self-attention, decoder self-attention, and cross-attention;
- prepare shifted decoder inputs and labels;
- train a tiny Transformer translation model;
- perform autoregressive decoding;
- explain pretrained translation checkpoints;
- distinguish MarianMT, M2M-100, and NLLB;
- explain target-language control;
- compare bilingual and multilingual MT;
- discuss zero-shot and fine-tuned translation;
- identify Arabic-specific tokenization and morphology issues.

## Table of Contents

1. Why Transformers Changed MT  
2. Encoder-Decoder Transformer  
3. Encoder Self-Attention  
4. Decoder Masked Self-Attention  
5. Cross-Attention  
6. Positional Information  
7. Feed-Forward Layers  
8. Residual Connections and Layer Normalization  
9. Shifted Decoder Inputs  
10. Training Objective  
11. Parallel Corpus  
12. Vocabulary  
13. Numericalization  
14. Padding and Masks  
15. Positional Encoding  
16. Tiny Transformer Model  
17. Training  
18. Training Curve  
19. Greedy Decoding  
20. Translation Examples  
21. Sequence Evaluation  
22. Beam Search  
23. Decoding Parameters  
24. Pretrained MT  
25. MarianMT  
26. M2M-100  
27. NLLB  
28. Bilingual versus Multilingual Models  
29. Language Codes  
30. Zero-Shot Translation  
31. Fine-Tuning  
32. Catastrophic Forgetting  
33. Domain Adaptation  
34. Arabic-English MT  
35. Tashkeel  
36. Morphology and Subwords  
37. Directionality  
38. Evaluation  
39. Error Analysis  
40. Transformer versus RNN NMT  
41. Computational Cost  
42. Optional Hugging Face Examples  
43. Reproducibility  
44. Knowledge Check  
45. Exercises  
46. Summary and Next Lesson

# 1. Why Transformers Changed MT

Transformer MT removes recurrence and processes sequence positions in parallel
during training.

This makes training more parallelizable while allowing direct long-range
interactions through attention.

In [ ]:
import math
import platform
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")

architectures = pd.DataFrame(
    [
        ("RNN NMT", "recurrent encoder and decoder"),
        ("Transformer MT", "self-attention + cross-attention"),
    ],
    columns=["Model family", "Core mechanism"],
)

architectures

# 2. Encoder-Decoder Transformer

A Transformer MT system has:

```text
source tokens -> encoder -> contextual source representations
                                 |
                                 v
target prefix -> decoder self-attention + cross-attention -> next-token logits
```

# 3. Encoder Self-Attention

Encoder tokens attend to other source tokens to build context-sensitive source
representations.

# 4. Decoder Masked Self-Attention

Decoder self-attention is causal: a target position must not see future target
tokens during training.

# 5. Cross-Attention

Decoder queries attend to encoder outputs.

This is the main mechanism through which target generation accesses source
information.

# 6. Positional Information

Self-attention alone is permutation invariant. Positional encodings or learned
positional embeddings therefore inject order information.

# 7. Feed-Forward Layers

Each Transformer block contains a position-wise feed-forward network after
attention.

# 8. Residual Connections and Layer Normalization

Residual paths and normalization stabilize deep Transformer training.

# 9. Shifted Decoder Inputs

During training:

```text
Target:         <sos> i am here <eos>
Decoder input:  <sos> i am here
Labels:         i am here <eos>
```

# 10. Training Objective

Transformer MT is usually trained with token-level cross-entropy over the target
sequence while ignoring padding positions.

# 11. Parallel Corpus

In [ ]:
parallel_pairs = [
    ("je suis ici", "i am here"),
    ("je suis petit", "i am small"),
    ("je suis grand", "i am big"),
    ("tu es ici", "you are here"),
    ("tu es petit", "you are small"),
    ("tu es grand", "you are big"),
    ("il est ici", "he is here"),
    ("il est petit", "he is small"),
    ("il est grand", "he is big"),
    ("elle est ici", "she is here"),
    ("elle est petite", "she is small"),
    ("elle est grande", "she is big"),
    ("nous sommes ici", "we are here"),
    ("nous sommes petits", "we are small"),
    ("vous etes ici", "you are here"),
    ("vous etes grands", "you are big"),
]

pd.DataFrame(
    parallel_pairs,
    columns=["French", "English"],
)

# 12. Vocabulary

In [ ]:
SPECIAL_TOKENS = ["<pad>", "<sos>", "<eos>", "<unk>"]
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

def build_vocab(sentences):
    lexical = sorted({
        token
        for sentence in sentences
        for token in sentence.split()
    })

    itos = SPECIAL_TOKENS + lexical
    stoi = {
        token: index
        for index, token
        in enumerate(itos)
    }

    return stoi, itos

source_sentences = [src for src, _ in parallel_pairs]
target_sentences = [tgt for _, tgt in parallel_pairs]

src_stoi, src_itos = build_vocab(source_sentences)
tgt_stoi, tgt_itos = build_vocab(target_sentences)

print("Source vocabulary:", len(src_itos))
print("Target vocabulary:", len(tgt_itos))

# 13. Numericalization

In [ ]:
def encode_sentence(sentence, stoi):
    ids = [SOS_IDX]

    ids.extend(
        stoi.get(token, UNK_IDX)
        for token in sentence.split()
    )

    ids.append(EOS_IDX)

    return torch.tensor(
        ids,
        dtype=torch.long,
    )

encode_sentence(
    "je suis ici",
    src_stoi,
)

# 14. Padding and Masks

In [ ]:
source_tensors = [
    encode_sentence(sentence, src_stoi)
    for sentence in source_sentences
]

target_tensors = [
    encode_sentence(sentence, tgt_stoi)
    for sentence in target_sentences
]

source_batch = pad_sequence(
    source_tensors,
    batch_first=True,
    padding_value=PAD_IDX,
).to(DEVICE)

target_batch = pad_sequence(
    target_tensors,
    batch_first=True,
    padding_value=PAD_IDX,
).to(DEVICE)

print(source_batch.shape)
print(target_batch.shape)

# 15. Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(
        self,
        d_model,
        max_length=128,
    ):
        super().__init__()

        positions = torch.arange(
            max_length
        ).unsqueeze(1)

        divisors = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
            )
            * (
                -math.log(10000.0)
                / d_model
            )
        )

        encoding = torch.zeros(
            max_length,
            d_model,
        )

        encoding[:, 0::2] = torch.sin(
            positions
            * divisors
        )

        encoding[:, 1::2] = torch.cos(
            positions
            * divisors
        )

        self.register_buffer(
            "encoding",
            encoding.unsqueeze(0),
        )

    def forward(self, x):
        return (
            x
            + self.encoding[
                :,
                :x.size(1),
                :
            ]
        )

# 16. Tiny Transformer Model

In [ ]:
class TinyTransformerMT(nn.Module):
    def __init__(
        self,
        source_vocab_size,
        target_vocab_size,
        d_model=48,
        nhead=4,
        encoder_layers=2,
        decoder_layers=2,
        dim_feedforward=96,
        dropout=0.0,
    ):
        super().__init__()

        self.d_model = d_model

        self.source_embedding = nn.Embedding(
            source_vocab_size,
            d_model,
            padding_idx=PAD_IDX,
        )

        self.target_embedding = nn.Embedding(
            target_vocab_size,
            d_model,
            padding_idx=PAD_IDX,
        )

        self.position = PositionalEncoding(
            d_model=d_model,
            max_length=64,
        )

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=encoder_layers,
            num_decoder_layers=decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )

        self.output = nn.Linear(
            d_model,
            target_vocab_size,
        )

    def causal_mask(
        self,
        target_length,
        device,
    ):
        return torch.triu(
            torch.full(
                (
                    target_length,
                    target_length,
                ),
                float("-inf"),
                device=device,
            ),
            diagonal=1,
        )

    def forward(
        self,
        source,
        decoder_input,
    ):
        source_padding_mask = (
            source == PAD_IDX
        )

        target_padding_mask = (
            decoder_input
            == PAD_IDX
        )

        target_mask = self.causal_mask(
            decoder_input.size(1),
            decoder_input.device,
        )

        source_embeddings = self.position(
            self.source_embedding(source)
            * math.sqrt(self.d_model)
        )

        target_embeddings = self.position(
            self.target_embedding(
                decoder_input
            )
            * math.sqrt(self.d_model)
        )

        hidden = self.transformer(
            src=source_embeddings,
            tgt=target_embeddings,
            tgt_mask=target_mask,
            src_key_padding_mask=source_padding_mask,
            tgt_key_padding_mask=target_padding_mask,
            memory_key_padding_mask=source_padding_mask,
        )

        return self.output(hidden)

# 17. Training

In [ ]:
model = TinyTransformerMT(
    source_vocab_size=len(src_itos),
    target_vocab_size=len(tgt_itos),
).to(DEVICE)

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX,
)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.005,
)

EPOCHS = 140
losses = []

decoder_input = target_batch[:, :-1]
labels = target_batch[:, 1:]

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()

    logits = model(
        source_batch,
        decoder_input,
    )

    loss = criterion(
        logits.reshape(
            -1,
            len(tgt_itos),
        ),
        labels.reshape(-1),
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0,
    )

    optimizer.step()

    losses.append(
        float(loss.item())
    )

print(
    "Final loss:",
    round(losses[-1], 4),
)

# 18. Training Curve

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(
    range(1, EPOCHS + 1),
    losses,
)
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Tiny Transformer MT Training")
plt.tight_layout()
plt.show()

# 19. Greedy Decoding

In [ ]:
def greedy_translate(
    sentence,
    max_new_tokens=10,
):
    model.eval()

    source = encode_sentence(
        sentence,
        src_stoi,
    ).unsqueeze(0).to(DEVICE)

    generated = torch.tensor(
        [[SOS_IDX]],
        dtype=torch.long,
        device=DEVICE,
    )

    with torch.no_grad():
        for _ in range(
            max_new_tokens
        ):
            logits = model(
                source,
                generated,
            )

            next_token = int(
                logits[
                    0,
                    -1
                ].argmax().item()
            )

            generated = torch.cat(
                [
                    generated,
                    torch.tensor(
                        [[next_token]],
                        dtype=torch.long,
                        device=DEVICE,
                    ),
                ],
                dim=1,
            )

            if (
                next_token
                == EOS_IDX
            ):
                break

    output_tokens = []

    for token_id in generated[
        0,
        1:
    ].tolist():
        if token_id == EOS_IDX:
            break

        if token_id not in {
            PAD_IDX,
            SOS_IDX,
        }:
            output_tokens.append(
                tgt_itos[token_id]
            )

    return " ".join(
        output_tokens
    )

greedy_translate(
    "je suis petit"
)

# 20. Translation Examples

In [ ]:
example_rows = []

pair_lookup = dict(
    parallel_pairs
)

for source in [
    "je suis ici",
    "tu es grand",
    "elle est petite",
    "il est petit",
    "nous sommes ici",
]:
    example_rows.append(
        {
            "source": source,
            "reference": (
                pair_lookup[source]
            ),
            "prediction": (
                greedy_translate(
                    source
                )
            ),
        }
    )

translation_frame = pd.DataFrame(
    example_rows
)

translation_frame

# 21. Sequence Evaluation

In [ ]:
evaluation_rows = []

for source, reference in parallel_pairs:
    prediction = greedy_translate(
        source
    )

    evaluation_rows.append(
        {
            "source": source,
            "reference": reference,
            "prediction": prediction,
            "exact_match": (
                prediction
                == reference
            ),
        }
    )

evaluation_frame = pd.DataFrame(
    evaluation_rows
)

print(
    "Exact sequence accuracy:",
    round(
        evaluation_frame[
            "exact_match"
        ].mean(),
        4,
    ),
)

# 22. Beam Search

Greedy decoding keeps one continuation. Beam search keeps several hypotheses and
scores complete sequences.

Important design choices include:

- beam width;
- length normalization;
- repetition penalties;
- early stopping.

# 23. Decoding Parameters

For deterministic translation, common generation controls include:

- maximum target length;
- number of beams;
- length penalty;
- forced target-language token;
- repetition constraints.

Sampling is generally less common for standard MT than for open-ended generation.

# 24. Pretrained MT

Pretrained MT models allow translation without training a model from scratch.

They may be:

- bilingual;
- multilingual;
- many-to-many;
- specialized for low-resource coverage.

# 25. MarianMT

MarianMT checkpoints are Transformer encoder-decoder translation models originally
trained with the Marian framework.

In Hugging Face, many OPUS checkpoints follow model names such as:

```text
Helsinki-NLP/opus-mt-en-de
```

Marian checkpoints are particularly convenient when a direct language-pair model
is available.

# 26. M2M-100

M2M-100 is a many-to-many multilingual translation family.

Instead of translating through English as a pivot, multilingual systems can model
direct language-to-language translation within one shared model.

Target-language control is explicitly supplied during generation.

# 27. NLLB

NLLB is a multilingual translation family designed to improve coverage of many
languages, including lower-resource languages.

Its language identifiers are more specific than simple two-letter language codes;
for example, English and Arabic can use script-aware identifiers such as
`eng_Latn` and `arb_Arab`.

# 28. Bilingual versus Multilingual Models

In [ ]:
model_families = pd.DataFrame(
    [
        (
            "MarianMT",
            "often bilingual or small multilingual groups",
            "direct language-pair checkpoints",
        ),
        (
            "M2M-100",
            "many-to-many multilingual",
            "one shared multilingual model",
        ),
        (
            "NLLB",
            "large multilingual coverage",
            "broad language coverage including lower-resource languages",
        ),
    ],
    columns=[
        "Family",
        "Typical scope",
        "Key idea",
    ],
)

model_families

# 29. Language Codes

Multilingual models need explicit source/target language identification.

Language-code mistakes can cause:

- wrong target language;
- tokenizer mismatch;
- invalid generation constraints.

# 30. Zero-Shot Translation

Zero-shot translation means applying a pretrained model to a language direction or
domain without task-specific fine-tuning on the current dataset.

Quality depends on the model's pretraining coverage and domain fit.

# 31. Fine-Tuning

Fine-tuning adapts a pretrained translation model to a parallel corpus.

Typical training data contains:

```text
source_text
target_text
```

# 32. Catastrophic Forgetting

Aggressive fine-tuning on a narrow domain may improve that domain while degrading
broader multilingual performance.

Validation should therefore include both in-domain and out-of-domain data when
generality matters.

# 33. Domain Adaptation

Fine-tuning is especially useful when terminology differs from the original
pretraining distribution.

Examples:

- legal;
- medical;
- scientific;
- technical support;
- literary translation.

# 34. Arabic-English MT

Arabic-English translation must account for:

- rich morphology;
- attached clitics;
- spelling variation;
- named entities;
- transliteration;
- optional tashkeel;
- word-order differences.

In [ ]:
arabic_examples = pd.DataFrame(
    [
        (
            "وَسَيَكْتُبُونَهَا",
            "complex verb with attached morphology",
        ),
        (
            "بِالْمَدْرَسَةِ",
            "preposition + definite noun",
        ),
        (
            "كِتَابُهُمَا",
            "noun + dual possessive suffix",
        ),
    ],
    columns=[
        "Arabic form",
        "MT consideration",
    ],
)

arabic_examples

# 35. Tashkeel

If the experiment is defined on fully vocalized Arabic, tashkeel should be
preserved consistently in:

- training;
- validation;
- test data;
- tokenizer inputs;
- references;
- model outputs;
- evaluation.

# 36. Morphology and Subwords

Subword models reduce out-of-vocabulary problems, but segmentation decisions still
affect morphology-sensitive translation.

Character-level approaches can preserve finer orthographic detail at the cost of
longer sequences.

# 37. Directionality

Arabic→English and English→Arabic are separate translation directions.

Their difficulty may differ because:

- target morphology differs;
- generation constraints differ;
- tokenization efficiency differs;
- ambiguity is asymmetric.

# 38. Evaluation

Modern MT evaluation should not rely on one metric.

Common choices include:

- BLEU;
- chrF / chrF++;
- COMET;
- semantic embedding similarity;
- human evaluation;
- statistical uncertainty.

# 39. Error Analysis

In [ ]:
error_taxonomy = pd.DataFrame(
    [
        ("Lexical", "wrong content word"),
        ("Morphology", "incorrect inflection"),
        ("Agreement", "gender/number/person mismatch"),
        ("Word order", "incorrect target ordering"),
        ("Omission", "source content missing"),
        ("Addition", "unsupported content generated"),
        ("Named entity", "name mistranslated"),
        ("Diacritization", "tashkeel incorrect or lost"),
    ],
    columns=[
        "Error type",
        "Description",
    ],
)

error_taxonomy

# 40. Transformer versus RNN NMT

In [ ]:
transformer_vs_rnn = pd.DataFrame(
    [
        (
            "Training sequence processing",
            "sequential",
            "parallel across positions",
        ),
        (
            "Long-range interaction",
            "through recurrent state",
            "direct via attention",
        ),
        (
            "Positional information",
            "implicit in recurrence",
            "must be added explicitly",
        ),
        (
            "Cross-source access",
            "attention over encoder states",
            "cross-attention",
        ),
    ],
    columns=[
        "Dimension",
        "RNN NMT",
        "Transformer MT",
    ],
)

transformer_vs_rnn

# 41. Computational Cost

Transformer self-attention has quadratic cost with sequence length in its standard
form.

Pretrained multilingual MT models can also be large enough that GPU inference is
substantially faster than CPU inference.

The tiny model in this notebook remains CPU-only.

# 42. Optional Hugging Face Examples

These cells are **examples only** and are disabled by default. They require model
downloads and may be slow on CPU.

In [ ]:
RUN_PRETRAINED_DEMOS = False

marian_example = '''
from transformers import pipeline

translator = pipeline(
    "translation_en_to_de",
    model="Helsinki-NLP/opus-mt-en-de",
    device=-1,
)

translator("Hello, how are you?")
'''

print(marian_example)

In [ ]:
m2m100_example = '''
from transformers import AutoTokenizer, M2M100ForConditionalGeneration

model_id = "facebook/m2m100_418M"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = M2M100ForConditionalGeneration.from_pretrained(model_id)

tokenizer.src_lang = "en"

inputs = tokenizer(
    "Life is like a box of chocolates",
    return_tensors="pt",
)

generated = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.get_lang_id("fr"),
)

translation = tokenizer.batch_decode(
    generated,
    skip_special_tokens=True,
)
'''

print(m2m100_example)

In [ ]:
nllb_example = '''
from transformers import pipeline

translator = pipeline(
    task="translation",
    model="facebook/nllb-200-distilled-600M",
    src_lang="eng_Latn",
    tgt_lang="arb_Arab",
    device=-1,
)

translator(
    "Machine translation connects people across languages."
)
'''

print(nllb_example)

The optional examples deliberately use `device=-1` for CPU execution. For larger
workloads, a compatible GPU can substantially reduce inference time.

# 43. Reproducibility

In [ ]:
reproducibility = pd.Series(
    {
        "module": (
            "Module 9 • Machine Translation"
        ),
        "lesson": (
            "Lesson 53 • Transformer-Based Machine Translation "
            "and Pretrained Multilingual Models"
        ),
        "sentence_pairs": (
            len(parallel_pairs)
        ),
        "source_vocab": (
            len(src_itos)
        ),
        "target_vocab": (
            len(tgt_itos)
        ),
        "epochs": (
            EPOCHS
        ),
        "device": str(
            DEVICE
        ),
        "seed": (
            SEED
        ),
        "python": (
            platform.python_version()
        ),
        "torch": (
            torch.__version__
        ),
        "offline_core": True,
    },
    name="Lesson 53 experiment",
)

reproducibility

# 44. Knowledge Check

1. Why does Transformer MT not need recurrent layers?
2. What does encoder self-attention do?
3. Why is decoder self-attention causal?
4. What does cross-attention connect?
5. Why are target inputs shifted during training?
6. What is greedy decoding?
7. What does beam search change?
8. What is MarianMT?
9. How does M2M-100 differ from a bilingual checkpoint?
10. What problem does target-language control solve?
11. What is zero-shot translation?
12. Why fine-tune a pretrained MT model?
13. What is catastrophic forgetting?
14. Why is Arabic morphology important for MT?
15. Why should fully vocalized experiments preserve tashkeel?

# 45. Exercises

## Exercise 1
Increase the tiny Transformer's number of layers.

## Exercise 2
Compare different attention-head counts.

## Exercise 3
Implement beam search.

## Exercise 4
Add label smoothing.

## Exercise 5
Train on a larger parallel corpus.

## Exercise 6
Compare RNN NMT and Transformer MT on the same toy data.

## Exercise 7
Run MarianMT for a supported language pair.

## Exercise 8
Compare M2M-100 and NLLB outputs.

## Exercise 9
Translate fully vocalized Arabic and inspect tashkeel preservation.

## Exercise 10
Fine-tune one pretrained translation model on a small domain dataset.

## Challenge Exercises

1. Add beam-search length normalization.
2. Train a character-level Transformer MT model.
3. Compare bilingual versus multilingual pretrained systems.
4. Fine-tune Arabic→English and English→Arabic separately.
5. Evaluate BLEU, chrF, COMET, and statistical confidence intervals.

# 46. Summary and Next Lesson

In this lesson:

- Transformer encoder-decoder MT was introduced;
- encoder self-attention, masked decoder self-attention, and cross-attention were
  distinguished;
- a complete tiny Transformer MT model was trained on CPU;
- greedy decoding and sequence evaluation were implemented;
- beam search and decoding controls were discussed;
- pretrained MT was introduced through MarianMT, M2M-100, and NLLB;
- bilingual versus multilingual modeling, language codes, zero-shot translation,
  fine-tuning, domain adaptation, catastrophic forgetting, Arabic morphology,
  directionality, and tashkeel preservation were covered.

## Next Lesson

**Lesson 54: Arabic–English Machine Translation — Morphology, Tokenization, and
Diacritization** focuses specifically on Arabic↔English translation, clitics,
segmentation, word/subword/character tokenization, tashkeel, ambiguity, and
direction-specific evaluation.

# References

- Vaswani, A. et al. *Attention Is All You Need*.
- Tiedemann, J. et al. work on OPUS and Marian translation models.
- Fan, A. et al. *Beyond English-Centric Multilingual Machine Translation*.
- Costa-jussà, M. et al. *No Language Left Behind*.
- Hugging Face Transformers documentation for MarianMT, M2M-100, and NLLB.